# Poster Clustering Notebook (Phase 4)

**Workflow**
1. Download posters for the ~5,000-film dataset (TMDb poster paths).
2. Convert posters into semantic visual vectors using **CLIP embeddings**.
3. Try **unsupervised clustering strategies** (PCA + KMeans variants).
4. Inspect clusters visually (collages + nearest-to-centroid examples).
5. Test relationships between visual clusters and variables such as:
   - LLM-derived fear categories
   - decades
   - production countries
6. Examine semantic patterns in poster clusters by analyzing the overview and keywords fields using **TF-IDF**


## Step 1 — Poster download

**Goal:** download all available poster images referenced in the cleaned dataset.

**Implementation**
- Build full URLs from TMDb’s `poster_path`.
- Save locally under `artifacts_posters/posters/`.
- Maintain a `_download_checkpoint.json` so the process can resume safely if it gets interrumped.

**Outcome**
- The resulting poster corpus (<= 5,000), which will be the input for CLIP embedding extraction in the next step.

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import json
import requests

#  - read the Phase 3 output CSV
#  - construct full TMDb URLs from relative poster paths
#  - save with the correct file extension (jpg)
#  - skip files already saved and keep a checkpoint of downloaded IDs
# 

# configuration

CSV_PATH        = Path("horror_data/horror_categorized_FULLSAMPLE_clean.csv")
OUT_DIR         = Path("artifacts_posters")
POSTERS_DIR     = OUT_DIR / "posters"
CHECKPOINT_JSON = OUT_DIR / "_download_checkpoint.json"
TMDB_BASE       = "https://image.tmdb.org/t/p/original"
COL_ID          = "id"
COL_POSTER      = "poster_path"

# creating the output directories as folders (after checking if they exist already)
OUT_DIR.mkdir(parents=True, exist_ok=True)
POSTERS_DIR.mkdir(parents=True, exist_ok=True)

# ---- load data ----
df = pd.read_csv(CSV_PATH)
df = df[df[COL_POSTER].notna()].copy()

# construct full URL and local path
df["poster_url"]  = TMDB_BASE.rstrip("/") + df[COL_POSTER]
df["poster_file"] = df[COL_ID].astype(str) + ".jpg"
df["local_path"]  = df["poster_file"].apply(lambda s: POSTERS_DIR / s)

# ---- load checkpoint ----
if CHECKPOINT_JSON.exists():
    try:
        checkpoint = set(json.loads(CHECKPOINT_JSON.read_text()).get("downloaded_ids", []))
    except Exception:
        checkpoint = set()
else:
    checkpoint = set()

# ---- download loop ----
downloaded_ids = set(checkpoint)
for _, row in tqdm(df.iterrows(), total=len(df), desc="Downloading posters"):
    mid = str(row[COL_ID])
    if mid in downloaded_ids:
        continue  # skip already done

    url = row["poster_url"]
    out_path = row["local_path"]

    # skip if file exists (from previous run)
    if out_path.exists():
        downloaded_ids.add(mid)
        continue

    try:
        r = requests.get(url, timeout=(5, 30))
        if r.status_code == 200:
            with open(out_path, "wb") as f:
                f.write(r.content)
            downloaded_ids.add(mid)
        else:
            print(f"HTTP {r.status_code} for {mid}")
    except Exception as e:
        print(f"Error downloading {mid}: {e}")

    # update checkpoint every 50 images
    if len(downloaded_ids) % 50 == 0:
        CHECKPOINT_JSON.write_text(json.dumps({"downloaded_ids": sorted(list(downloaded_ids))}, indent=2))

# ---- final checkpoint save ----
CHECKPOINT_JSON.write_text(json.dumps({"downloaded_ids": sorted(list(downloaded_ids))}, indent=2))

print(f"Done. Posters OK: {len(downloaded_ids)} / {len(df)}")
print(f"Saved under: {POSTERS_DIR.resolve()}")

### Download results

**Result:** 4,928 posters downloaded successfully (out of 4,928 with valid poster paths).

## Step 2 — Visual embeddings with CLIP (feature extraction)

**Goal:** represent poster images as embeddings that capture semantic visual information.

**Model:** `openai/clip-vit-base-patch32`  
- Output: a **512-dimensional embedding** per poster.

**Output:**
- `artifacts_posters/clip_embeddings.npy` — embedding matrix (N x 512)
- `artifacts_posters/embeddings_index.json` — metadata mapping each embedding row to `{movie_id, file_name}`

In [ ]:
# CLIP embeddings extraction for posters
# --------------------------------------------
# - reads all image files in POSTERS_DIR
# - uses the CLIP model to transform each poster image into a high-dimensional embedding vector that captures semantic visual content
# - outputs:
#     artifacts_posters/clip_embeddings.npy  
#     artifacts_posters/embeddings_index.json (metadata list: movie_id, file_name)

import torch
import numpy as np
from PIL import Image
from pathlib import Path
from transformers import CLIPProcessor, CLIPModel
import json
from tqdm import tqdm

# ----------------------
# config
# ----------------------
# model_name selects the exact CLIP variant (on this case vit-base-patch32)
# posters_dir is the folder where the image files are
# out_dir is where all resulting artifacts will be saved
model_name = "openai/clip-vit-base-patch32"
POSTERS_DIR = Path("artifacts_posters/posters")
OUT_DIR = Path("artifacts_posters")
EMBEDDINGS_FILE = OUT_DIR / "clip_embeddings.npy"
EMBEDDINGS_INDEX = OUT_DIR / "embeddings_index.json"

# creating output directory if missing ensures the script works on clean environments
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------
# load model & processor
# ----------------------
print("Loading CLIP model...")

device = "cuda" if torch.cuda.is_available() else "cpu"
# CLIPModel loads the pretrained weights
model = CLIPModel.from_pretrained(model_name).to(device)

# CLIPProcessor handles all preprocessing (resize, normalization, etc.) consistently
processor = CLIPProcessor.from_pretrained(model_name)

# ----------------------
# helper: single-image embedding
# ----------------------
def extract_clip_embedding(image_path: Path) -> np.ndarray | None:
    """
    extract CLIP image embedding for a single poster.

    returns:
        a normalized embedding vector (1d numpy array) or None if something fails
    """
    try:
        image = Image.open(image_path).convert("RGBA")

        # processor applies CLIP-specific transforms and returns tensors ready for the model
        inputs = processor(images=image, return_tensors="pt").to(device)

        # CLIP's image encoder converts image to semantic vector
        image_features = model.get_image_features(**inputs)
        image_features = torch.nn.functional.normalize(image_features, p=2, dim=-1)

        embedding = image_features.detach().cpu().numpy().reshape(-1)

        return embedding

    except Exception as e:
        # errors are caught instead of breaking the pipeline
        print(f"Error processing {image_path}: {e}")
        return None

# ----------------------
# main: loop over posters
# ----------------------
print("Collecting poster files...")

# gathering all possible image types
poster_files = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
    poster_files.extend(POSTERS_DIR.glob(ext))

poster_files = sorted(poster_files)

print(f"Found {len(poster_files)} poster files.")

embeddings_list = []
embeddings_index = []

for poster_path in tqdm(poster_files, desc="Extracting CLIP embeddings"):
    emb = extract_clip_embedding(poster_path)
    if emb is None:
        # skipping missing/failed embeddings avoids breaking the alignment between features and metadata
        continue

    # append the embedding
    embeddings_list.append(emb)

    # storing metadata (movie_id and file_name) lets you map rows to movies after saving the matrix
    embeddings_index.append({
        "movie_id": poster_path.stem,
        "file_name": poster_path.name,
    })

# ----------------------
# save outputs
# ----------------------
# sanity check: if no embeddings, something is wrong
if len(embeddings_list) == 0:
    raise RuntimeError("No embeddings were generated. Check POSTERS_DIR and file types.")

# vstack converts a list of 1d arrays into a single 2d matrix
embeddings_matrix = np.vstack(embeddings_list)

np.save(EMBEDDINGS_FILE, embeddings_matrix)

# json preserving row to file metadata alignment
with open(EMBEDDINGS_INDEX, "w") as f:
    json.dump(embeddings_index, f, indent=2)

print(f"Saved embeddings matrix to: {EMBEDDINGS_FILE}")
print(f"Matrix shape: {embeddings_matrix.shape}")
print(f"Saved index JSON to:       {EMBEDDINGS_INDEX}")

### Embedding extraction results

**Result:** embeddings generated for the 4,928 posters, producing a matrix of shape (4928, 512)

This embedding matrix is the numerical representation we will use for dimensionality reduction and clustering.

## Step 3 — Clustering experiments (PCA + KMeans)

CLIP vectors are 512D, which can be too high to interpret.  
I apply **PCA** to compress embeddings while preserving as much variance as possible, then run **KMeans** on the result of the PCA.

### Experiments:
KMeans requires choosing:
- `PCA_DIM` (how many dimensions to keep)
- `K` (how many clusters)

So I ran multiple variations to analyze the resulting cluster collages and see which one returns the most interpretable groups.

For each `(PCA_DIM, K)` pair I save:
- PCA-reduced embeddings (`clip_embeddings_pca{DIM}d.npy`)
- cluster assignments (`embeddings_with_clusters.json`)
- collage images (`cluster_galleries/cluster_<id>.jpg`)
- saved PCA + KMeans models (`.joblib`) for reproducibility

In [ ]:
# ============================================================
# EXPERIMENTS WITH KMEANS CLUSTERING ON PCA-REDUCED EMBEDDINGS
# ============================================================

import json
import random
from math import ceil
from collections import defaultdict
from pathlib import Path

import numpy as np
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import joblib

# ------------------------------------------------------------
# global paths shared by all experiments
# ------------------------------------------------------------

# base_dir is the root folder where everything related to posters lives
BASE_DIR = Path("artifacts_posters")

# posters_dir holds the original poster image files (jpg/png/webp)
POSTERS_DIR = BASE_DIR / "posters"

# clip_emb_path is the single .npy file with 512-dimensional CLIP embeddings
CLIP_EMB_PATH = BASE_DIR / "clip_embeddings.npy"

# global_index_json is the master JSON where each row of CLIP_EMB_PATH
# is mapped to some metadata such as {movie_id, file_name}
GLOBAL_INDEX_JSON = BASE_DIR / "embeddings_index.json"


# ------------------------------------------------------------
# helper: build collage images for each cluster
# ------------------------------------------------------------
def build_cluster_galleries(
    index_with_clusters,
    posters_dir: Path,
    cluster_gallery_dir: Path,
    max_per_cluster: int = 36,
    thumb_w: int = 256,
    thumb_h: int = 384,
    grid_cols: int = 6,
):
    """
    given a list of metadata records that already have a "cluster" field,
    this function creates one collage image per cluster.

    each collage is composed of thumbnails of posters that belong to that cluster.
    the collages are saved as "cluster_<id>.jpg" inside cluster_gallery_dir.
    """

    # make sure the gallery folder exists so saves don't fail
    cluster_gallery_dir.mkdir(parents=True, exist_ok=True)

    # group all records by cluster id in a dict: cluster_id -> list[record]
    clusters = defaultdict(list)
    for info in index_with_clusters:
        # we use get(..., -1) so missing "cluster" defaults to -1
        cid = info.get("cluster", -1)
        # we treat cluster -1 as "noise" and skip it
        if cid == -1:
            continue
        clusters[cid].append(info)

    # now we iterate over each cluster and create one collage per cluster
    for cid, items in clusters.items():
        # in case a cluster has many posters, we cap the number of thumbnails
        # to keep the collage readable and the file size reasonable
        if len(items) <= max_per_cluster:
            sample = items
        else:
            sample = random.sample(items, max_per_cluster)

        n = len(sample)

        # compute how many rows we need in the grid based on how many
        # thumbnails we will place and how many columns per row
        rows = ceil(n / grid_cols)

        # compute total collage width and height in pixels
        gallery_w = grid_cols * thumb_w
        gallery_h = rows * thumb_h

        # create a blank black canvas where all thumbnails will be pasted
        gallery = Image.new("RGB", (gallery_w, gallery_h), color=(0, 0, 0))

        # loop over each poster sample and paste it in the correct grid cell
        for i, info in enumerate(sample):
            poster_path = posters_dir / info["file_name"]
            if not poster_path.exists():
                # if the file is missing we skip it
                continue

            # open the image and ensure it is in RGB mode
            img = Image.open(poster_path).convert("RGB")

            # resize the image in-place so that it fits inside the thumbnail box
            # while preserving aspect ratio
            img.thumbnail((thumb_w, thumb_h), Image.LANCZOS)

            # create a fixed-size thumbnail canvas so every slot has the same size
            thumb = Image.new("RGB", (thumb_w, thumb_h), color=(0, 0, 0))

            # center the resized image within that thumbnail box
            offset_x = (thumb_w - img.width) // 2
            offset_y = (thumb_h - img.height) // 2
            thumb.paste(img, (offset_x, offset_y))

            # compute the grid coordinates: column and row index
            col = i % grid_cols
            row = i // grid_cols

            # convert grid coordinates into pixel coordinates
            x = col * thumb_w
            y = row * thumb_h

            # paste the thumbnail into the collage canvas
            gallery.paste(thumb, (x, y))

        # build the output path for this cluster's collage
        out_path = cluster_gallery_dir / f"cluster_{cid}.jpg"

        # save the collage as a jpg file
        gallery.save(out_path, quality=90)
        print(f"    saved gallery for cluster {cid} -> {out_path}")


# ------------------------------------------------------------
# single kmeans + PCA experiment
# ------------------------------------------------------------
def run_kmeans_experiment(PCA_DIM: int, K: int):
    """
    runs one complete experiment with the following steps:
    ...
    """

    experiment_name = f"kmeans_k{K}_pca{PCA_DIM}"

    # all the artifacts for this experiment live under this folder
    exp_dir = BASE_DIR / "experiments" / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    # path where we will save the experiment-specific index with cluster labels
    index_with_clusters_path = exp_dir / "embeddings_with_clusters.json"

    # path where we will save the pca-reduced embeddings
    pca_embeddings_path = exp_dir / f"clip_embeddings_pca{PCA_DIM}d.npy"

    # >>> NEW: paths for the saved models
    pca_model_path    = exp_dir / f"pca_{PCA_DIM}d.joblib"
    kmeans_model_path = exp_dir / f"kmeans_k{K}_pca{PCA_DIM}.joblib"
    # <<< NEW

    # folder where we will store collage images, one per cluster
    cluster_gallery_dir = exp_dir / "cluster_galleries"
    cluster_gallery_dir.mkdir(parents=True, exist_ok=True)

    print("\n====================================================")
    print("kmeans experiment:", experiment_name)
    print("  pca_dim:", PCA_DIM, "k:", K)
    print("  output folder:", exp_dir)

    # -----------------------
    # step 1: load embeddings and index
    # -----------------------

    emb = np.load(CLIP_EMB_PATH)

    with GLOBAL_INDEX_JSON.open() as f:
        index = json.load(f)

    # -----------------------
    # step 2: apply pca
    # -----------------------

    reducer = PCA(n_components=PCA_DIM, random_state=42)

    emb_pca = reducer.fit_transform(emb)

    # save the reduced embeddings
    np.save(pca_embeddings_path, emb_pca)
    print("  saved pca embeddings to:", pca_embeddings_path)

    # >>> NEW: save the fitted PCA model
    joblib.dump(reducer, pca_model_path)
    print("  saved PCA model to:", pca_model_path)
    # <<< NEW

    # -----------------------
    # step 3: kmeans clustering
    # -----------------------

    kmeans = KMeans(n_clusters=K, random_state=14)

    labels = kmeans.fit_predict(emb_pca)

    # >>> NEW: save the fitted KMeans model
    joblib.dump(kmeans, kmeans_model_path)
    print("  saved KMeans model to:", kmeans_model_path)
    # <<< NEW

    # -----------------------
    # step 4: attach labels to metadata
    # -----------------------

    for rec, lab in zip(index, labels):
        rec["cluster"] = int(lab)

    # -----------------------
    # step 5: save index + galleries
    # -----------------------

    with index_with_clusters_path.open("w") as f:
        json.dump(index, f, indent=2, ensure_ascii=False)
    print("  wrote enriched metadata with clusters to:", index_with_clusters_path)

    build_cluster_galleries(index, POSTERS_DIR, cluster_gallery_dir)


In [ ]:
# ------------------------------------------------------------
# list of kmeans + pca variants to run
# ------------------------------------------------------------
EXPERIMENTS = [
    # experiment 1: 240 pca, 20 clusters
    {"PCA_DIM": 240, "K": 20},

    # experiment 2: 280 pca, 50 clusters
    {"PCA_DIM": 280, "K": 50},

    # experiment 3: 120 pca, 20 clusters
    {"PCA_DIM": 120, "K": 20},

    # experiment 4: 160 pca, 40 clusters
    {"PCA_DIM": 160, "K": 40},

    # experiment 5: 280 pca, 40 clusters
    {"PCA_DIM": 280, "K": 40},

]

# ------------------------------------------------------------
# run all defined experiments
# ------------------------------------------------------------

for cfg in EXPERIMENTS:
    # unpack the dictionary keys to arguments and call the experiment function
    run_kmeans_experiment(**cfg)

From the tested runs, I selected the experiment `kmeans_k50_pca280` as the basis for the curated clusters analyzed in later steps.

## Step 4 — Manual curation of interpretable clusters

After inspecting cluster collages, I created a curated subset of clusters that:
- had clear visual coherence
- corresponded to some recognizable theme (e.g., religious horror, creature features, etc.)
- seemed promising for cross-analysis with decades, countries, or other relevant variable

This curation is saved as `artifacts_posters/experiments/selected_clusters.json`

In [ ]:
import json
import numpy as np
from pathlib import Path

# saving the clusters I want to keep for further analysis in a JSON format:
selected_clusters_data = {
  "experiment_name": "kmeans_k50_pca280",
  "clusters": [
    { "cluster_id": 1,  "label": "Shark & Aquatic Monster Horror",                         "collage_file": "cluster_1.jpg" },
    { "cluster_id": 3,  "label": "Modern Creature Feature Horror",                         "collage_file": "cluster_3.jpg" },
    { "cluster_id": 8,  "label": "Vintage Camp & Creature Schlock (1960s-80s)",            "collage_file": "cluster_8.jpg" },
    { "cluster_id": 9,  "label": "Modern Isolation & Survival Horror",                     "collage_file": "cluster_9.jpg" },
    { "cluster_id": 14, "label": "Classic Monster Movies (1950s-1960s)",                   "collage_file": "cluster_14.jpg" },
    { "cluster_id": 16, "label": "Japanese Vintage Horror",                    "collage_file": "cluster_16.jpg" },
    { "cluster_id": 22, "label": "Modern Female-Led Psychological Horror",                 "collage_file": "cluster_22.jpg" },
    { "cluster_id": 25, "label": "Rural / Backwoods Survival Horror",                      "collage_file": "cluster_25.jpg" },
    { "cluster_id": 28, "label": "Vintage Illustrated Monster Movies (Pre-1980)",          "collage_file": "cluster_28.jpg" },
    { "cluster_id": 30, "label": "Supernatural Gothic Horror (1990s-2000s)",               "collage_file": "cluster_30.jpg" },
    { "cluster_id": 31, "label": "Latin American Horror (Vintage)",              "collage_file": "cluster_31.jpg" },
    { "cluster_id": 39, "label": "Modern Monster & Creature Horror",                       "collage_file": "cluster_39.jpg" },
    { "cluster_id": 41, "label": "Vintage Gothic (mix)",                   "collage_file": "cluster_41.jpg" },
    { "cluster_id": 42, "label": "Religious Horror",                      "collage_file": "cluster_42.jpg" },
    { "cluster_id": 43, "label": "Modern Neon / Synthwave",                         "collage_file": "cluster_43.jpg" },
    { "cluster_id": 46, "label": "B-Movie Creature Features & Kaiju",                      "collage_file": "cluster_46.jpg" },
    { "cluster_id": 47, "label": "Modern Teen / Ensemble Horror-Comedy",                   "collage_file": "cluster_47.jpg" },
    { "cluster_id": 48, "label": "South Asian Horror",     "collage_file": "cluster_48.jpg" }
  ]
}

SAVE_PATH = Path("/Users/lara/dataviz-s1/machine-learning/Project/artifacts_posters/experiments/selected_clusters.json")
SAVE_PATH.write_text(json.dumps(selected_clusters_data, indent=2), encoding="utf-8")


## Step 5 — Cluster "prototypes" and nearest posters

To better visualize/understand each selected cluster, I compute a **prototype vector**:

Process:
- For each curated cluster: take all member embeddings
- Normalize embeddings
- Compute the mean direction
- Normalize again

Then I get the **16 closest posters** to each prototype.


In [ ]:
# now we want to define the centers for the selected clusters,
# to get one 512-dimensional vector per cluster, which becomes the "prototype" of that cluster

import json
import numpy as np
from pathlib import Path

# paths
EXP_DIR = Path("/Users/lara/dataviz-s1/machine-learning/Project/artifacts_posters/experiments/kmeans_k50_pca280")
SELECTED_JSON = Path("/Users/lara/dataviz-s1/machine-learning/Project/artifacts_posters/experiments/selected_clusters.json")

# 1) load PCA embeddings for the chosen experiment
# these are the PCA-reduced embeddings (shape: N x 280), each row is a poster
embeddings = np.load(EXP_DIR / "clip_embeddings_pca280d.npy")

# 2) load metadata with cluster labels
# this json includes movie_id, file_name, and cluster
with open(EXP_DIR / "embeddings_with_clusters.json") as f:
    emb_meta = json.load(f)

# 3) load curated list of cluster IDs (manually selected)
with open(SELECTED_JSON) as f:
    selected_ids = {c["cluster_id"] for c in json.load(f)["clusters"]}

# 4) compute normalized mean vector per curated cluster (this will be the "prototype" for each cluster)
cluster_prototypes = {}

for cid in selected_ids:
    # indices of rows that belong to this cluster
    # emb_meta[i]["cluster"] tells us the original KMeans label
    idxs = [i for i, row in enumerate(emb_meta) if row.get("cluster") == cid]

    # get their embeddings (PCA 280d)
    vecs = embeddings[idxs]                        
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)  # normalize each
    # this ensures that when we average, we are averaging directions on the unit sphere and not magnitudes
    # it forces every embedding to have length 1

    center = vecs.mean(axis=0)                     # mean in normalized space
    # this gives the average direction of all embeddings in this cluster
    # that direction represents the "prototype" of the cluster

    center = center / np.linalg.norm(center)       # normalize the prototype itself
    # after averaging vectors, we normalize again

    # and save
    cluster_prototypes[cid] = center

# 5) save prototypes (dict: cluster_id -> 280d vector)
np.save(EXP_DIR / "cluster_prototypes_pca280d.npy", cluster_prototypes)

print("Saved:", EXP_DIR / "cluster_prototypes_pca280d.npy")
print("Clusters computed:", len(cluster_prototypes))


In [ ]:
# analyzing the selected clusters by finding the top 16 closest posters to each cluster prototype
 
import numpy as np, json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# load everything directly from paths
emb = np.load("/Users/lara/dataviz-s1/machine-learning/Project/artifacts_posters/experiments/kmeans_k50_pca280/clip_embeddings_pca280d.npy")
with open("/Users/lara/dataviz-s1/machine-learning/Project/artifacts_posters/experiments/kmeans_k50_pca280/embeddings_with_clusters.json") as f:
    meta = json.load(f)
protos = np.load("/Users/lara/dataviz-s1/machine-learning/Project/artifacts_posters/experiments/kmeans_k50_pca280/cluster_prototypes_pca280d.npy", allow_pickle=True).item()
with open("/Users/lara/dataviz-s1/machine-learning/Project/artifacts_posters/experiments/selected_clusters.json") as f:
    labels = {c["cluster_id"]: c["label"] for c in json.load(f)["clusters"]}

POSTERS_DIR = Path("/Users/lara/dataviz-s1/machine-learning/Project/artifacts_posters/posters")

# normalize embeddings once for cosine similarity
emb_norm = emb / np.linalg.norm(emb, axis=1, keepdims=True)

def top16(cid):
    proto = protos[cid] / np.linalg.norm(protos[cid])
    sims = emb_norm @ proto
    idx = sims.argsort()[::-1][:16]
    return [(meta[i]["file_name"], float(sims[i])) for i in idx]

# plot 2 rows of 8 posters per cluster (16 total)
for cid in sorted(protos.keys()):
    title = f"Cluster {cid} — {labels.get(cid,'')}"
    print("\n" + title)

    items = top16(cid)

    fig, axes = plt.subplots(2, 8, figsize=(16, 6))
    fig.suptitle(title, fontsize=14)

    for i, (fname, score) in enumerate(items):
        row = i // 8
        col = i % 8
        axes[row, col].imshow(Image.open(POSTERS_DIR / fname))
        axes[row, col].set_title(f"{score:.2f}", fontsize=8)
        axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()

### Takeaway

Some clusters show strong alignment with specific fear categories.
For example, the cluster "Shark & Aquatic Monster Horror" is dominated by Ecological / Natural Menace, "Religious Horror" concentrates in Possession & Loss of Agency, and "Classic Monster Movies" is mostly Invasion & Paranoia.

Other clusters are more mixed, suggesting that poster aesthetics are determined by a mix of features, what was to be expected.